# 3-Projekte-Investitionsproblem mit QAOA

Dieses Notebook löst ein kleines Investitionsproblem mit Qiskit, QAOA und SamplerV2.

**Budget:** 7 Mio. €

| Projekt | Kosten | Gewinn |
|---|---:|---:|
| A | 4 Mio. € | 9 |
| B | 3 Mio. € | 7 |
| C | 2 Mio. € | 4 |

Gesucht wird die Kombination mit maximalem Gewinn bei höchstens 7 Mio. € Kosten.

In [ ]:
# Installation
# In Google Colab einmal ausführen.

%pip install -U qiskit qiskit-aer qiskit-algorithms qiskit-optimization

In [ ]:
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer

from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA

from qiskit_aer.primitives import SamplerV2

In [ ]:
# Projektdaten

projects = {
    "A": {"cost": 4, "profit": 9},
    "B": {"cost": 3, "profit": 7},
    "C": {"cost": 2, "profit": 4}
}

budget = 7

for name, data in projects.items():
    print(
        f"Projekt {name}: "
        f"Kosten = {data['cost']} Mio. €, "
        f"Gewinn = {data['profit']}"
    )
        
print(f"Budget = {budget} Mio. €")

In [ ]:
# ======================================================
# Optimierungsproblem definieren
# ======================================================

problem = QuadraticProgram(
    name="Investitionsproblem"
)

# Binärvariablen:
# 0 = Projekt nicht auswählen
# 1 = Projekt auswählen

for project in projects:
    problem.binary_var(name=project)

# Gewinn maximieren
problem.maximize(
    linear={
        project: data["profit"]
        for project, data in projects.items()
    }
)

# Budgetbeschränkung
problem.linear_constraint(
    linear={
        project: data["cost"]
        for project, data in projects.items()
    },
    sense="<=",
    rhs=budget,
    name="Budget"
)

print(problem)

## QAOA konfigurieren

Wir verwenden den modernen `SamplerV2` von Qiskit Aer als lokalen Simulator.

In [ ]:
# SamplerV2
sampler = SamplerV2()

# QAOA
qaoa = QAOA(
    sampler=sampler,
    optimizer=COBYLA(maxiter=100),
    reps=2
)

# High-Level-Verbindung zwischen
# QuadraticProgram und QAOA
optimizer = MinimumEigenOptimizer(qaoa)

In [ ]:
# ======================================================
# Problem lösen
# ======================================================

result = optimizer.solve(problem)

print(result)

In [ ]:
# ======================================================
# Ergebnis auswerten
# ======================================================

selected = []
total_cost = 0
total_profit = 0

for project, value in zip(projects, result.x):
    if round(value) == 1:
        selected.append(project)
        total_cost += projects[project]["cost"]
        total_profit += projects[project]["profit"]

print("=" * 50)
print("QAOA ERGEBNIS")
print("=" * 50)

print("Ausgewählte Projekte:", " + ".join(selected))
print(f"Gesamtkosten: {total_cost} Mio. €")
print(f"Gesamtgewinn: {total_profit}")
print(f"Budget: {budget} Mio. €")

if total_cost <= budget:
    print("✓ Budget eingehalten")
else:
    print("✗ Budget überschritten")

## Erwartete optimale Lösung

Die optimale Kombination ist:

**Projekt A + Projekt B**

- Kosten: 4 + 3 = **7 Mio. €**
- Gewinn: 9 + 7 = **16**

Die drei Entscheidungsvariablen können als drei Qubits interpretiert werden:

`A = 1, B = 1, C = 0`

entspricht der Auswahl von A und B.